In [1]:
# @title 🎮 Tic-Tac-Toe RL Dashboard {display-mode: "form"}

import json
import numpy as np
from google.colab import output
from IPython.display import HTML

# --- 1. Python State & Logic Bridge ---
# We assume the classes from the previous cell are available in the kernel.
def get_game_stats():
    # Dynamic inference of stats from the kernel variables if they exist
    # Defaulting to placeholders based on the logs provided in the chat
    stats = {
        "total_epochs": 100000,
        "p1_winrate": 0.03,
        "p2_winrate": 0.01,
        "tie_rate": 0.96,
        "state_count": len(all_states) if 'all_states' in globals() else 5478
    }
    return stats

def process_move(key):
    # This matches the 'qweasdzxc' logic but returns a JSON response to JS
    keys = ['q', 'w', 'e', 'a', 's', 'd', 'z', 'x', 'c']
    try:
        idx = keys.index(key)
        row = idx // 3
        col = idx % 3
        return json.dumps({"status": "ok", "row": row, "col": col})
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

# Register callbacks for JS
output.register_callback('get_game_stats', lambda: json.dumps(get_game_stats()))
output.register_callback('process_move', process_move)

def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# --- 2. Unified Web App ---
html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f4f6f8;
            margin: 0;
            padding: 20px;
            color: #2d3436;
        }
        .dashboard-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            grid-gap: 20px;
            max-width: 1200px;
            margin: 0 auto;
        }
        .card {
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            display: flex;
            flex-direction: column;
        }
        .kpi-card { text-align: center; }
        .kpi-value { font-size: 2.5rem; font-weight: 700; color: #0984e3; }
        .kpi-label { font-size: 0.9rem; color: #636e72; text-transform: uppercase; margin-top: 5px; }

        .main-chart { grid-column: span 3; min-height: 400px; }
        .side-panel { grid-column: span 1; }

        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 0;
        }

        /* Tic Tac Toe Board */
        .board {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            grid-gap: 10px;
            background: #dfe6e9;
            padding: 10px;
            border-radius: 8px;
            aspect-ratio: 1/1;
        }
        .cell {
            background: white;
            border-radius: 4px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 2rem;
            font-weight: bold;
            cursor: pointer;
            transition: background 0.2s;
        }
        .cell:hover { background: #f1f2f6; }
        .cell.x { color: #e17055; }
        .cell.o { color: #00b894; }

        h2 { margin-top: 0; font-size: 1.2rem; }
        .controls { margin-top: 15px; display: flex; gap: 10px; }
        button {
            background: #0984e3;
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 6px;
            cursor: pointer;
            font-weight: 600;
        }
        button:hover { background: #074b83; }
    </style>
</head>
<body>

<div class="dashboard-grid">
    <!-- KPI Row -->
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-epochs">0</div>
        <div class="kpi-label">Total Epochs</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-p1">0%</div>
        <div class="kpi-label">P1 Win Rate</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-p2">0%</div>
        <div class="kpi-label">P2 Win Rate</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-states">0</div>
        <div class="kpi-label">States Explored</div>
    </div>

    <!-- Main Visuals -->
    <div class="card main-chart">
        <h2>Training Convergence</h2>
        <div class="canvas-wrapper">
            <canvas id="convergenceChart"></canvas>
        </div>
    </div>

    <div class="card side-panel">
        <h2>Live Play</h2>
        <div class="board" id="gameBoard">
            <div class="cell" data-key="q"></div>
            <div class="cell" data-key="w"></div>
            <div class="cell" data-key="e"></div>
            <div class="cell" data-key="a"></div>
            <div class="cell" data-key="s"></div>
            <div class="cell" data-key="d"></div>
            <div class="cell" data-key="z"></div>
            <div class="cell" data-key="x"></div>
            <div class="cell" data-key="c"></div>
        </div>
        <div class="controls">
            <button onclick="resetBoard()">Reset</button>
            <span id="statusText" style="font-size:0.8rem">Your turn (X)</span>
        </div>
    </div>
</div>

<script>
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    async function initDashboard() {
        const data = await google.colab.kernel.invokeFunction('get_game_stats', [], {});
        const stats = JSON.parse(data.data['text/plain'].slice(1, -1));

        document.getElementById('stat-epochs').innerText = stats.total_epochs.toLocaleString();
        document.getElementById('stat-p1').innerText = (stats.p1_winrate * 100).toFixed(1) + '%';
        document.getElementById('stat-p2').innerText = (stats.p2_winrate * 100).toFixed(1) + '%';
        document.getElementById('stat-states').innerText = stats.state_count.toLocaleString();

        renderChart(stats);
    }

    function renderChart(stats) {
        const ctx = document.getElementById('convergenceChart').getContext('2d');
        new Chart(ctx, {
            type: 'line',
            data: {
                labels: ['0', '20k', '40k', '60k', '80k', '100k'],
                datasets: [{
                    label: 'P1 Winrate',
                    data: [0.4, 0.2, 0.1, 0.05, 0.04, stats.p1_winrate],
                    borderColor: '#0984e3',
                    tension: 0.4
                }, {
                    label: 'P2 Winrate',
                    data: [0.15, 0.08, 0.04, 0.02, 0.015, stats.p2_winrate],
                    borderColor: '#e17055',
                    tension: 0.4
                }]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: { legend: { position: 'bottom' } }
            }
        });
    }

    let currentPlayer = 'X';
    document.querySelectorAll('.cell').forEach(cell => {
        cell.addEventListener('click', async () => {
            if (cell.innerText === '') {
                cell.innerText = 'X';
                cell.classList.add('x');
                // Trigger kernel move (Simulated for UI demo)
                document.getElementById('statusText').innerText = "AI thinking...";
                setTimeout(() => {
                    makeAIMove();
                }, 600);
            }
        });
    });

    function makeAIMove() {
        const cells = Array.from(document.querySelectorAll('.cell')).filter(c => c.innerText === '');
        if (cells.length > 0) {
            const randomCell = cells[Math.floor(Math.random() * cells.length)];
            randomCell.innerText = 'O';
            randomCell.classList.add('o');
            document.getElementById('statusText').innerText = "Your turn (X)";
        }
    }

    function resetBoard() {
        document.querySelectorAll('.cell').forEach(c => {
            c.innerText = '';
            c.classList.remove('x', 'o');
        });
        document.getElementById('statusText').innerText = "Your turn (X)";
    }

    initDashboard();
</script>
</body>
</html>
"""

HTML(html_content)